In [312]:
import os
from dotenv import load_dotenv
from pathlib import Path
# from dataclasses import dataclass
from typing import List, Dict, Any

import xml.etree.ElementTree as ET

In [313]:

BASE_DIR = Path.cwd().parents[0]


load_dotenv(BASE_DIR / "creds" / ".env")




True

In [314]:
def xml_to_dict(element) -> Dict[str, Any]:
    """Convert XML element to clean dictionary."""
    result = {}
    
    # Add attributes if they exist
    if element.attrib:
        result['_attributes'] = element.attrib
    
    # Add text content
    if element.text and element.text.strip():
        result['_text'] = element.text.strip()
    
    # Process child elements
    for child in element:
        child_data = xml_to_dict(child)
        
        # Handle multiple elements with same tag
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_data)
        else:
            # Single element - just store the data
            result[child.tag] = child_data if child_data else (child.text or None)
    
    return result if result else None


def parse_vehicle_search(xml_string: str) -> Dict:
    """Parse vehicle search response into clean data."""
    root = ET.fromstring(xml_string)
    
    # Extract header info
    header = root.find('Header')
    response_data = {
        'status': header.findtext('Status') if header is not None else None,
        'status_code': header.findtext('StatusCode') if header is not None else None,
        'vehicles': []
    }
    
    # Extract vehicle data
    for vehicle_elem in root.findall('.//VehicleSearchItem'):
        # Extract all links
        links = []
        links_container = vehicle_elem.find('Links')
        if links_container is not None:
            for link in links_container.findall('Link'):
                links.append({
                    'href': link.findtext('Href'),
                    'rel': link.findtext('Rel')
                })
        
        vehicle = {
            'base_vehicle_id': vehicle_elem.findtext('BaseVehicleID'),
            'make_name': vehicle_elem.findtext('MakeName'),
            'model_name': vehicle_elem.findtext('ModelName'),
            'sub_model_name': vehicle_elem.findtext('SubModelName'),
            'year': vehicle_elem.findtext('Year'),
            'engine_description': vehicle_elem.findtext('EngineDescription'),
            'vehicle_id': vehicle_elem.findtext('VehicleID'),
            'is_active': vehicle_elem.findtext('VehicleIsActive') == 'true',
            'links': links
        }
        response_data['vehicles'].append(vehicle)
    
    return response_data

<h2>Set Up</h2>
<p>
Run this first to set up the required functions:
</p>

In [315]:
from datetime import datetime, timezone
from urllib.parse import urlparse
import urllib
import hmac
import hashlib
import requests
import base64

C_PUBLIC_KEY = os.getenv("C_PUBLIC_KEY")
C_PRIVATE_KEY = os.getenv("C_PRIVATE_KEY")


if (not C_PUBLIC_KEY) | (not C_PRIVATE_KEY):
    raise EnvironmentError("API keys not set") 

def GenerateSharedAuth(d: datetime, public_key: str, private_key: str, uri: str, http_verb: str) -> bytes:
    utc_time = d.astimezone(timezone.utc)
    epoch_time = int(utc_time.timestamp())
    relative_url = urlparse(uri).path
    encoded_url = urllib.parse.quote(relative_url)

    plain_sig = public_key + chr(10) + http_verb + chr(10) + str(epoch_time) + chr(10) + encoded_url
    key = private_key.encode('ascii')
    byte_sig = plain_sig.encode('ascii')

    signature = hmac.new(key, byte_sig, hashlib.sha256).digest()
    return signature

def GetResponse(uri: str) -> str:
    today = datetime.now()
    auth = GenerateSharedAuth(today, C_PUBLIC_KEY, C_PRIVATE_KEY, uri, "GET")
    headers = {"Authorization": "Shared " + C_PUBLIC_KEY + ":" + base64.b64encode(auth).decode('ascii'), 
               "Date": today.astimezone(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT"),
               "Host": "api.motor.com"}

    response = requests.get("https://api.motor.com" + uri, headers=headers)
    return response.text

In [316]:


def extract_keywords_from_xml(xml_string: str, debug: bool = False) -> Dict:
    """
    Extract status, status_code, ApplicationID, and DisplayName from any XML response.
    Returns application_ids as a list of dicts with id and display_name.
    Handles nested ApplicationIDs and XML namespaces.
    Strips quotes from extracted values.
    """
    root = ET.fromstring(xml_string)

    # Strip namespace from tag names for easier searching
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    def strip_quotes(value):
        """Remove surrounding quotes from string values."""
        if value and isinstance(value, str):
            return value.strip('\'"')
        return value

    # Extract status and status_code
    result = {
        'status': None,
        'status_code': None,
        'application_ids': []  # List of dicts with id and display_name
    }

    # Search for ApplicationID/DisplayName pairs within same parent
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = strip_quotes(elem.text)
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = strip_quotes(elem.text)
        elif tag == 'EstimatedWorkTimeApplicationSummary' or tag == 'ApplicationSummary':
            # Extract paired ApplicationID and DisplayName from same parent
            app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
            display_name_elem = next((e for e in elem if strip_ns(e.tag) == 'DisplayName'), None)

            if app_id_elem is not None and app_id_elem.text:
                app_id = strip_quotes(app_id_elem.text)
                display_name = strip_quotes(display_name_elem.text) if display_name_elem is not None and display_name_elem.text else None
                
                # Append as dict to list
                result['application_ids'].append({
                    'id': app_id,
                    'display_name': display_name if display_name else 'Unknown'
                })

    if debug:
        print(f"Found {len(result['application_ids'])} ApplicationID(s)")

    return result


def extract_application_id(xml_string: str) -> str:
    """
    Extract ApplicationID from any XML response.
    Returns the first ApplicationID found.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    # Find first ApplicationID element
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'ApplicationID' and elem.text:
            return elem.text
    
    return None


def extract_all_part_application_ids(xml_string: str, debug: bool = False) -> List[str]:
    """
    Extract ALL part ApplicationIDs from a parts-summary response.
    Returns a list of all ApplicationID values found within PartApplicationSummary/PartApp containers.
    """
    root = ET.fromstring(xml_string)

    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag

    ids = []

    # Look for ApplicationID within PartApplicationSummary/PartApp containers
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'PartApplicationSummary' or tag == 'PartApp':
            # Look for ApplicationID as direct child
            for child in elem:
                child_tag = strip_ns(child.tag)
                if child_tag == 'ApplicationID' and child.text:
                    ids.append(child.text)
                    if debug:
                        print(f"  Found ApplicationID: {child.text}")
                    break  # Only take first ApplicationID per container

    if debug:
        print(f"Total ApplicationIDs found: {len(ids)}")

    return ids


def extract_estimated_work_time(xml_string: str) -> Dict:
    """
    Extract EstimatedWorkTime details from XML response.
    Returns status, status_code, and a details dict with labor time and skill information.
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find EstimatedWorkTime element
    work_time_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'EstimatedWorkTime'), None)
    
    if work_time_elem is not None:
        # Extract Job Description from Notes > Note > Text
        job_description = None
        notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
        if notes_elem is not None:
            note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
            if note_elem is not None:
                job_description = get_text(note_elem, 'Text')
        
        # Extract RequiredSkill > Description
        required_skill = None
        skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
        if skill_elem is not None:
            required_skill = get_text(skill_elem, 'Description')
        
        # Build details dict
        result['details'] = {
            'job_description': job_description,
            'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
            'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
            'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
            'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
            'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
            'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
            'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
            'required_skill': required_skill,
            'service_type': get_text(work_time_elem, 'ServiceType'),
            'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
            'is_active': get_text(work_time_elem, 'IsActive'),
            'type': get_text(work_time_elem, 'Type')
        }
    
    return result


def extract_part_details(xml_string: str) -> Dict:
    """
    Extract part details from XML response.
    Returns status, status_code, and a details dict with part information.
    Handles nested structure: Part > PricingFamilies > PartPricingFamily > Pricing > PartPricing
    """
    root = ET.fromstring(xml_string)
    
    def strip_ns(tag):
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Helper to get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    result = {
        'status': None,
        'status_code': None,
        'details': {}
    }
    
    # Extract status and status_code
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag == 'Status' and elem.text:
            result['status'] = elem.text
        elif tag == 'StatusCode' and elem.text:
            result['status_code'] = elem.text
    
    # Find Part element
    part_elem = next((e for e in root.iter() if strip_ns(e.tag) == 'Part'), None)
    
    if part_elem is not None:
        part_number = get_text(part_elem, 'PartNumber')
        
        # Navigate to PartPricingFamily
        country_name = None
        manufacturer_name = None
        effective_date = None
        is_current = None
        oepr_part_number = None
        motor_part_number = None
        net_core_price = None
        category_name = None
        part_terminology_name = None
        price = None
        return_old_part = None
        
        pricing_families = next((e for e in part_elem if strip_ns(e.tag) == 'PricingFamilies'), None)
        if pricing_families is not None:
            part_pricing_family = next((e for e in pricing_families if strip_ns(e.tag) == 'PartPricingFamily'), None)
            if part_pricing_family is not None:
                # Extract CountryInfo > Name
                country_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'CountryInfo'), None)
                if country_elem is not None:
                    country_name = get_text(country_elem, 'Name')
                
                # Extract ManufacturerInfo > Name
                manufacturer_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'ManufacturerInfo'), None)
                if manufacturer_elem is not None:
                    manufacturer_name = get_text(manufacturer_elem, 'Name')
                
                # Navigate to Pricing > PartPricing
                pricing_elem = next((e for e in part_pricing_family if strip_ns(e.tag) == 'Pricing'), None)
                if pricing_elem is not None:
                    part_pricing = next((e for e in pricing_elem if strip_ns(e.tag) == 'PartPricing'), None)
                    if part_pricing is not None:
                        effective_date = get_text(part_pricing, 'EffectiveDate')
                        is_current = get_text(part_pricing, 'IsCurrent')
                        oepr_part_number = get_text(part_pricing, 'OEPRPartNumber')
                        motor_part_number = get_text(part_pricing, 'MOTORPartNumber')
                        net_core_price = get_text(part_pricing, 'NetCorePrice')
                        price = get_text(part_pricing, 'Price')
                        return_old_part = get_text(part_pricing, 'ReturnOldPart')
                        
                        # Extract Category > Name from PCDBPart
                        pcdb_part = next((e for e in part_pricing if strip_ns(e.tag) == 'PCDBPart'), None)
                        if pcdb_part is not None:
                            category_elem = next((e for e in pcdb_part if strip_ns(e.tag) == 'Category'), None)
                            if category_elem is not None:
                                category_name = get_text(category_elem, 'Name')
                            part_terminology_name = get_text(pcdb_part, 'PartTerminologyName')
        
        # Build details dict
        result['details'] = {
            'part_number': part_number,
            'country_name': country_name,
            'manufacturer_name': manufacturer_name,
            'effective_date': effective_date,
            'is_current': is_current,
            'oepr_part_number': oepr_part_number,
            'motor_part_number': motor_part_number,
            'net_core_price': net_core_price,
            'category_name': category_name,
            'part_terminology_name': part_terminology_name,
            'price': price,
            'return_old_part': return_old_part
        }
    
    return result

In [317]:
# # Test the function
# result = extract_keywords_from_xml(resp)

# print("Extracted Keywords:")
# print(f"  Status: {result['status']}")
# print(f"  Status Code: {result['status_code']}")
# print(f"  Application ID: {result['application_id']}")
# print(f"  All Application IDs: {result['application_ids']}")

# # Convert to JSON
# print("\nAs JSON:")
# print(json.dumps({
#     'status': result['status'],
#     'status_code': result['status_code'],
#     'application_id': result['application_id']
# }, indent=2))

<h2>Get Vehicle Info by VIN</h2>

<p>/v1/Information/Vehicles/Search/ByVIN?vin={VIN}</p>

In [318]:
vin = "1FTEW1E45KFB21693"

veh_vin = {
            "US":{""
                    "escape_2014": "3FA6P0HD1ER388009",
                    "escape_2020": "3FA6P0D9XLR115438"},
            "CA":{
                "escape_2014": "1FMCU9G97EUB92197",
                "escape_2025": "1FMCU9NZXSUA08739"}
            }

In [319]:

def step_1(Vin):
    resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={Vin}")
    # resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")

    data = parse_vehicle_search(resp)

    print(f"Response Status: {data['status']} ({data['status_code']})\n")
    return data

<h2>Get Summary</h2>
<p>/v1/Information/Vehicles/Attributes/BaseVehicleId/{VehId}/Content/Summaries/Of/EstimatedWorkTimes</p>

In [320]:
def step2(data):

   vehicle_id = data["vehicles"][0]["base_vehicle_id"]
   system_id = "5"
   group_id = ""
   sub_group_id = ""
   # print(vehicle_id)
   
   resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID={group_id}&subGroupID={sub_group_id}")


   # print(resp)
   if not resp:
      raise ValueError("API response fails - step 2")
   
   data = extract_keywords_from_xml(resp)
   data["vehicle_id"] = vehicle_id
   return data




In [321]:
def step3(data):
    """
    REFACTORED: Extract ALL work items organized as list of dicts.
    Takes output from step2 (contains vehicle_id and application_ids list).
    
    For each work-time application, loops through ALL EstimatedWorkTime items
    and creates separate dict objects for each. Returns a list where each element
    is a dict mapping service name to list of work items.
    
    Output structure:
    [
        {
            'Rack & Pinion Assembly R&R': [
                {
                    'application_id': '244560030',
                    'vehicle_id': '85215',
                    'job_description': 'Includes: The removal...',
                    'base_labor_time': '0.4',
                    'base_labor_time_description': 'One Side',
                    'all_labor_time': '0.4',
                    'all_labor_time_description': 'One Side',
                    'all_warranty_labor_time': '0',
                    'base_warranty_labor_time': '0',
                    'additional_labor_time': '0',
                    'additional_labor_time_description': '',
                    'additional_warranty_labor_time': '0',
                    'estimated_work_time_id': '10613',
                    'labor_time_interval': 'Hours',
                    'required_skill': 'Requires a person...',
                    'service_type': 'Service',
                    'base_labor_time_average': '0',
                    'is_active': 'false',
                    'type': 'Main Operation'
                },
                {
                    'application_id': '244560030',
                    'vehicle_id': '85215',
                    'job_description': 'Includes: The removal...',
                    'base_labor_time': '0.7',
                    'base_labor_time_description': 'Both Sides',
                    ... (second work item)
                }
            ]
        },
        {
            'Steering Knuckle R&R': [
                { work_item_1 },
                { work_item_2 },
                ...
            ]
        }
    ]
    """
    def strip_ns(tag):
        """Remove XML namespace from tag."""
        return tag.split('}')[-1] if '}' in tag else tag
    
    def get_text(elem, tag_name):
        """Get text content from child element."""
        child = next((e for e in elem if strip_ns(e.tag) == tag_name), None)
        return child.text if child is not None and child.text else None
    
    vehicle_id = data["vehicle_id"]
    application_ids = data["application_ids"]  # List of {id, display_name}
    
    result_list = []  # Master list to hold all service dicts
    
    # Loop through each work-time application
    for app in application_ids:
        app_id = app["id"]
        display_name = app["display_name"]
        
        # Get the response which may contain multiple EstimatedWorkTime items
        resp = GetResponse(
            f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
            f"/Content/Details/Of/EstimatedWorkTimes/{app_id}"
        )
        
        root = ET.fromstring(resp)
        
        # Find the Items container with all EstimatedWorkTime elements
        items_container = None
        for elem in root.iter():
            if strip_ns(elem.tag) == 'Items':
                items_container = elem
                break
        
        # Collect all work items for this service
        work_items_list = []
        
        if items_container is not None:
            for work_time_elem in items_container:
                if strip_ns(work_time_elem.tag) == 'EstimatedWorkTime':
                    # Extract required skill description
                    required_skill = None
                    skill_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'RequiredSkill'), None)
                    if skill_elem is not None:
                        required_skill = get_text(skill_elem, 'Description')
                    
                    # Extract job description from Notes > Note > Text
                    job_description = None
                    notes_elem = next((e for e in work_time_elem if strip_ns(e.tag) == 'Notes'), None)
                    if notes_elem is not None:
                        note_elem = next((e for e in notes_elem if strip_ns(e.tag) == 'Note'), None)
                        if note_elem is not None:
                            job_description = get_text(note_elem, 'Text')
                    
                    # Create a dict for this work item with all fields
                    work_item = {
                        'application_id': app_id,
                        'vehicle_id': vehicle_id,
                        'job_description': job_description,
                        'additional_labor_time': get_text(work_time_elem, 'AdditionalLaborTime'),
                        'additional_labor_time_description': get_text(work_time_elem, 'AdditionalLaborTimeDescription'),
                        'additional_warranty_labor_time': get_text(work_time_elem, 'AdditionalWarrantyLaborTime'),
                        'all_labor_time': get_text(work_time_elem, 'AllLaborTime'),
                        'all_labor_time_description': get_text(work_time_elem, 'AllLaborTimeDescription'),
                        'all_warranty_labor_time': get_text(work_time_elem, 'AllWarrantyLaborTime'),
                        'base_labor_time': get_text(work_time_elem, 'BaseLaborTime'),
                        'base_labor_time_description': get_text(work_time_elem, 'BaseLaborTimeDescription'),
                        'base_warranty_labor_time': get_text(work_time_elem, 'BaseWarrantyLaborTime'),
                        'estimated_work_time_id': get_text(work_time_elem, 'EstimatedWorkTimeID'),
                        'labor_time_interval': get_text(work_time_elem, 'LaborTimeInterval'),
                        'required_skill': required_skill,
                        'service_type': get_text(work_time_elem, 'ServiceType'),
                        'base_labor_time_average': get_text(work_time_elem, 'BaseLaborTimeAverage'),
                        'is_active': get_text(work_time_elem, 'IsActive'),
                        'type': get_text(work_time_elem, 'Type')
                    }
                    
                    # Add this work item to the list
                    work_items_list.append(work_item)
        
        # Create dict: {service_name: [list of work items]}
        service_dict = {display_name: work_items_list}
        
        # Add to master list
        result_list.append(service_dict)
    
    return result_list

In [322]:
def step_4(labor_items):
    """
    Enrich each work item with its parts data.
    Input: list of dicts where each dict is {service_name: [work_items]}
    Output: single dict with service_name: [work_items_with_parts]
    
    Parts are stored as a dict keyed by part_app_id (not a list).
    
    Flattened structure:
    {
        'Rack & Pinion Assembly R&R': [
            {
                'application_id': '244560030',
                ...labor fields...,
                'parts': {
                    'part_app_id_1': {
                        'part_number': 'KG9Z 3504-H',
                        'price': '2718.18',
                        ...
                    },
                    'part_app_id_2': {...}
                }
            }
        ],
        'Steering Knuckle R&R': [...]
    }
    """
    enriched_dict = {}
    
    # Flatten the list of dicts into a single dict
    for service_dict in labor_items:
        for display_name, work_items in service_dict.items():
            enriched_work_items = []
            
            for work_item in work_items:
                application_id = work_item['application_id']
                vehicle_id = work_item['vehicle_id']
                
                # Get parts summary for this work-time
                resp = GetResponse(
                    f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                    f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
                )
                
                # Extract all part application IDs
                part_app_ids = extract_all_part_application_ids(resp)
                
                # Get details for each part - store as dict keyed by part_app_id
                parts_dict = {}
                for part_app_id in part_app_ids:
                    resp = GetResponse(
                        f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                        f"/Content/Details/Of/Parts/{part_app_id}"
                    )
                    part_details = extract_part_details(resp)
                    
                    part_info = part_details.get('details', {})
                    parts_dict[part_app_id] = part_info
                
                enriched_work_item = {
                    **work_item,
                    'parts': parts_dict
                }
                
                enriched_work_items.append(enriched_work_item)
            
            enriched_dict[display_name] = enriched_work_items
    
    return enriched_dict

In [323]:
"""
UPDATED step_5: Returns a LIST of part details (not a single dict)
This is critical - step_5 MUST return a list for the pipeline to work.
"""
def step_5(data):
    """
    Get details for ALL parts (loop through all part_app_ids).
    IMPORTANT: Returns a LIST of dicts, NOT a single dict.
    """
    vehicle_id = data.get("vehicle_id")
    part_app_ids = data.get("part_app_ids", [])
    
    # Initialize result as a LIST
    all_parts_list = []
    
    # Loop through each part ID and get its details
    for part_app_id in part_app_ids:
        try:
            resp = GetResponse(
                f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
                f"/Content/Details/Of/Parts/{part_app_id}"
            )

            part_details = extract_part_details(resp)
            
            # Append to list
            all_parts_list.append({
                'part_app_id': part_app_id,
                'part_data': part_details
            })
        except Exception as e:
            print(f"    Error getting part {part_app_id}: {e}")
    
    # CRITICAL: Return the list, not a single dict
    return all_parts_list



In [324]:
veh_data = step_1(veh_vin["US"]["escape_2020"])



Response Status: OK (200)



In [325]:

veh_data
 

{'status': 'OK',
 'status_code': '200',
 'vehicles': [{'base_vehicle_id': '85215',
   'make_name': 'Ford',
   'model_name': 'Fusion',
   'sub_model_name': 'Titanium',
   'year': '2020',
   'engine_description': '2.0L L4 (9) Turbocharged GAS FI',
   'vehicle_id': '179683',
   'is_active': True,
   'links': [{'href': '/v1/Information/Vehicles/Attributes/BaseVehicleID/85215/BaseVehicle',
     'rel': 'BaseVehicleDetails'},
    {'href': '/v1/Information/Vehicles/Attributes/VehicleID/179683/Vehicle',
     'rel': 'VehicleDetails'}]}]}

In [326]:

labour_summary_items = step2(veh_data)
 

In [327]:
labour_summary_items

{'status': 'OK',
 'status_code': '200',
 'application_ids': [{'id': '244560030',
   'display_name': 'Rack & Pinion Assembly R&R'},
  {'id': '244549997', 'display_name': 'Steering Knuckle R&R'},
  {'id': '244550120', 'display_name': 'Steering Knuckle R&R'},
  {'id': '244579908', 'display_name': 'Tie Rod R&R'},
  {'id': '244580012', 'display_name': 'Tie Rod R&R'}],
 'vehicle_id': '85215'}

In [328]:
labour_items = step3(labour_summary_items)

In [329]:
len(labour_items[4])

1

In [330]:
resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Details/Of/EstimatedWorkTimes/{244560030}")


In [331]:

# print(resp)
 

In [332]:
# resp_4 = step_4(labour_details)

In [333]:
# resp = GetResponse(f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{85215}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{244579908}")
 

In [334]:
# print(resp)

In [335]:


# step_5(labour_more_details)

In [336]:
# labour_details["application_id"]

In [337]:
# labour_more_details = step_4(labour_details)

In [338]:
# step_5(labour_more_details)

In [339]:
# def run_pipeline(vin: str) -> List[Dict]:
#     """
#     Full pipeline function: takes a VIN and returns all parts information 
#     for every work-time application and related part.
    
#     Returns a list of dicts, one entry per (work-time × part) combination.
#     Each entry contains work-time metadata, labor details, and part details.
#     """
#     # Step 1: VIN lookup
#     resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
#     vehicle_data = parse_vehicle_search(resp)
#     base_vehicle_id = vehicle_data["vehicles"][0]["base_vehicle_id"]
    
#     # Step 2: Get all work-time application IDs
#     resp = GetResponse(
#         f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}"
#         f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID=5&groupID=&subGroupID="
#     )
#     summaries = extract_keywords_from_xml(resp)
    
#     results = []
    
#     # Step 3: Loop over each work-time application
#     for app in summaries["application_ids"]:
#         app_id = app["id"]
#         display_name = app["display_name"]
        
#         # Get labor details
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/EstimatedWorkTimes/{app_id}"
#         )
#         labor = extract_estimated_work_time(resp)
        
#         # Get related part application IDs
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{app_id}"
#         )
#         part_app_ids = extract_all_part_application_ids(resp)
        
#         # Get details for each part
#         for part_app_id in part_app_ids:
#             resp = GetResponse(
#                 f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{base_vehicle_id}/Content/Details/Of/Parts/{part_app_id}"
#             )
#             part = extract_part_details(resp)
            
#             results.append({
#                 'work_time_application_id': app_id,
#                 'work_time_display_name': display_name,
#                 'labor_details': labor.get('details', {}),
#                 'part_details': part.get('details', {})
#             })
    
#     return results
# results = run_pipeline(veh_vin["US"]["escape_2020"])
# results

In [340]:
# def run_pipeline_v2(vin: str) -> Dict:
#     """
#     Pipeline that follows the exact same strategy as step1→step2→step3→step4→step5.
#     Loops through all application IDs and all parts, stores results by vehicle_id.
#     Uses modified step_4 and step_5 that handle lists of part IDs.
#     """
#     print(f"Starting pipeline for VIN: {vin}\n")
    
#     # STEP 1: Vehicle VIN lookup
#     print("STEP 1: Vehicle lookup...")
#     resp = GetResponse(f"/v1/Information/Vehicles/Search/ByVIN?vin={vin}")
#     data = parse_vehicle_search(resp)
#     print(f"✓ Vehicle found\n")
    
#     # STEP 2: Get all work-time application IDs
#     print("STEP 2: Get work-time summaries...")
#     vehicle_id = data["vehicles"][0]["base_vehicle_id"]
#     system_id = "5"
#     resp = GetResponse(
#         f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#         f"/Content/Summaries/Of/EstimatedWorkTimes/?systemID={system_id}&groupID=&subGroupID="
#     )
#     app_ID = extract_keywords_from_xml(resp)
#     app_ID["vehicle_id"] = vehicle_id
#     print(f"✓ Found {len(app_ID['application_ids'])} work-time applications\n")
    
#     # Storage for all results organized by vehicle_id
#     all_results = {
#         vehicle_id: {
#             'vehicle_info': data["vehicles"][0],
#             'parts_data': []
#         }
#     }
    
#     # STEP 3-5: Loop through EACH application_id and ALL its parts
#     print("STEPS 3-5: Processing each application and its parts...\n")
#     for app_idx, app in enumerate(app_ID['application_ids'], 1):
#         application_id = app['id']
#         display_name = app['display_name']
        
#         print(f"  {app_idx}. Processing: {display_name} (ID: {application_id})")
        
#         # STEP 3: Get labor details
#         labour_details_data = {
#             'vehicle_id': vehicle_id,
#             'application_id': application_id,
#             'display_name': display_name
#         }
#         resp = GetResponse(
#             f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#             f"/Content/Details/Of/EstimatedWorkTimes/{application_id}"
#         )
#         labour_data = extract_estimated_work_time(resp)
#         labour_details_data.update(labour_data)
#         print(f"     ✓ Got labor details")
        
#         # STEP 4: Get parts summary (get ALL part application IDs)
#         step4_result = step_4(labour_details_data)
#         part_app_ids = step4_result.get("part_app_ids", [])
#         print(f"     ✓ Found {len(part_app_ids)} parts")
        
#         # STEP 5: Loop through EACH part and get details
#         parts_list = step_5(step4_result)
        
#         # Ensure parts_list is actually a list
#         if not isinstance(parts_list, list):
#             print(f"     ⚠️  Step 5 returned {type(parts_list)}, expected list")
#             parts_list = []
        
#         for part_idx, part_result in enumerate(parts_list, 1):
#             # Handle both old and new data structures
#             if isinstance(part_result, dict):
#                 part_number = part_result.get('part_data', {}).get('details', {}).get('part_number', 'N/A')
#                 part_app_id = part_result.get('part_app_id')
#                 part_details = part_result.get('part_data', {}).get('details', {})
#             else:
#                 part_number = 'N/A'
#                 part_app_id = 'N/A'
#                 part_details = {}
            
#             # Store combined result
#             result_entry = {
#                 'work_time': {
#                     'application_id': application_id,
#                     'display_name': display_name,
#                     'labor_details': labour_data.get('details', {})
#                 },
#                 'part': {
#                     'application_id': part_app_id,
#                     'details': part_details
#                 }
#             }
#             all_results[vehicle_id]['parts_data'].append(result_entry)
#             print(f"       {part_idx}. ✓ {part_number}")
        
#         print()
    
#     total_parts = len(all_results[vehicle_id]['parts_data'])
#     print(f"\n✓ Pipeline complete!")
#     print(f"  Vehicle ID: {vehicle_id}")
#     print(f"  Total work-times: {len(app_ID['application_ids'])}")
#     print(f"  Total parts: {total_parts}\n")
    
#     return all_results

# # Test with the working VIN
# results_v2 = run_pipeline_v2(veh_vin["US"]["escape_2020"])
# print(f"\nFinal result structure: {list(results_v2.keys())}")

In [341]:
# results_v2["85215"]["parts_data"]

In [342]:
# # CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
# print("="*80)
# print("DEBUGGING: XML Element Structure Analysis")
# print("="*80 + "\n")

# vehicle_id = "85215"
# application_id = "244560030"

# resp = GetResponse(
#     f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#     f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
# )

# root = ET.fromstring(resp)

# def strip_ns(tag):
#     return tag.split('}')[-1] if '}' in tag else tag

# # Step 1: Find all PartApplicationSummary elements
# print("Step 1: Looking for PartApplicationSummary elements...\n")
# part_app_summaries = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary':
#         part_app_summaries.append(elem)
#         print(f"✓ Found PartApplicationSummary")
#         print(f"  Full tag (with namespace): '{elem.tag}'")
#         print(f"  Stripped tag: '{tag}'")
        
#         # Check its children
#         print(f"  Direct children:")
#         child_count = 0
#         for child in elem:
#             child_count += 1
#             child_tag = strip_ns(child.tag)
#             print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
#             # If it's ApplicationID, show it
#             if child_tag == 'ApplicationID':
#                 print(f"        → FOUND ApplicationID: {child.text}")
        
#         if child_count == 0:
#             print(f"    (no direct children)")
#         print()

# print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# # Step 2: Try the extraction with detailed steps
# print("Step 2: Simulating extract_all_part_application_ids logic...\n")
# ids = []
# containers_found = 0
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary' or tag == 'PartApp':
#         containers_found += 1
#         print(f"Found container: {tag}")
        
#         # Look for ApplicationID child
#         for e in elem:
#             e_tag = strip_ns(e.tag)
#             if e_tag == 'ApplicationID':
#                 print(f"  ✓ Found child ApplicationID: {e.text}")
#                 if e.text:
#                     ids.append(e.text)
        
#         # Also try with next() like in the function
#         app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
#         if app_id_elem is not None:
#             print(f"  next() also found: {app_id_elem.text}")
#         else:
#             print(f"  next() returned None")

# print(f"\nTotal containers found: {containers_found}")
# print(f"Total IDs extracted: {ids}\n")

# # Step 3: Show what extract_application_id would find
# print("Step 3: What extract_application_id finds...\n")
# app_ids = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'ApplicationID' and elem.text:
#         app_ids.append(elem.text)
#         print(f"Found ApplicationID: {elem.text}")

# print(f"Total ApplicationID elements: {len(app_ids)}\n")

<h2>Get Details</h2>
<p>/Vehicles/Attributes/BaseVehicleId/{VehicleId}/Content/Details/Of/EstimatedWorkTimes/{ApplicationId}</p>

<h2>Whatever link you want</h2>

In [343]:
resp = GetResponse("/v1/Information/Vehicles/Attributes/BaseVehicleId/81898/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/185575044")
print(resp)

<?xml version="1.0"?>
<MWSPartSummaryRs xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <Body>
    <Attributes>
      <Countries>
        <VehicleCountry>
          <Code>USA</Code>
          <CountryID>1</CountryID>
          <Name>United States</Name>
        </VehicleCountry>
      </Countries>
      <Engines>
        <EngineInfo>
          <Aspiration>Turbocharged</Aspiration>
          <BlockType>V</BlockType>
          <CID>183</CID>
          <CylinderCC>2993</CylinderCC>
          <CylinderHeadType>DOHC</CylinderHeadType>
          <CylinderLiter>3.0</CylinderLiter>
          <Cylinders>6</Cylinders>
          <Description>3.0L V6 (1) Turbocharged DIESEL FI</Description>
          <Designation>-</Designation>
          <EngineBoreInch>3.31</EngineBoreInch>
          <EngineBoreMetric>84.0</EngineBoreMetric>
          <EngineID>13215</EngineID>
          <EngineStrokeInch>3.54</EngineStrokeInch>
          <EngineStrokeMetric>

In [344]:
# # CRITICAL DEBUG: Understand why extract_all_part_application_ids fails
# print("="*80)
# print("DEBUGGING: XML Element Structure Analysis")
# print("="*80 + "\n")

# vehicle_id = "85215"
# application_id = "244560030"

# resp = GetResponse(
#     f"/v1/Information/Vehicles/Attributes/BaseVehicleId/{vehicle_id}"
#     f"/Content/Summaries/Of/Parts/RelatedTo/EstimatedWorkTimes/{application_id}"
# )

# root = ET.fromstring(resp)

# def strip_ns(tag):
#     return tag.split('}')[-1] if '}' in tag else tag

# # Step 1: Find all PartApplicationSummary elements
# print("Step 1: Looking for PartApplicationSummary elements...\n")
# part_app_summaries = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary':
#         part_app_summaries.append(elem)
#         print(f"✓ Found PartApplicationSummary")
#         print(f"  Full tag (with namespace): '{elem.tag}'")
#         print(f"  Stripped tag: '{tag}'")
        
#         # Check its children
#         print(f"  Direct children:")
#         child_count = 0
#         for child in elem:
#             child_count += 1
#             child_tag = strip_ns(child.tag)
#             print(f"    [{child_count}] {child_tag:30s} = {child.text[:50] if child.text else 'None'}")
            
#             # If it's ApplicationID, show it
#             if child_tag == 'ApplicationID':
#                 print(f"        → FOUND ApplicationID: {child.text}")
        
#         if child_count == 0:
#             print(f"    (no direct children)")
#         print()

# print(f"Total PartApplicationSummary elements found: {len(part_app_summaries)}\n")

# # Step 2: Try the extraction with detailed steps
# print("Step 2: Simulating extract_all_part_application_ids logic...\n")
# ids = []
# containers_found = 0
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'PartApplicationSummary' or tag == 'PartApp':
#         containers_found += 1
#         print(f"Found container: {tag}")
        
#         # Look for ApplicationID child
#         for e in elem:
#             e_tag = strip_ns(e.tag)
#             if e_tag == 'ApplicationID':
#                 print(f"  ✓ Found child ApplicationID: {e.text}")
#                 if e.text:
#                     ids.append(e.text)
        
#         # Also try with next() like in the function
#         app_id_elem = next((e for e in elem if strip_ns(e.tag) == 'ApplicationID'), None)
#         if app_id_elem is not None:
#             print(f"  next() also found: {app_id_elem.text}")
#         else:
#             print(f"  next() returned None")

# print(f"\nTotal containers found: {containers_found}")
# print(f"Total IDs extracted: {ids}\n")

# # Step 3: Show what extract_application_id would find
# print("Step 3: What extract_application_id finds...\n")
# app_ids = []
# for elem in root.iter():
#     tag = strip_ns(elem.tag)
#     if tag == 'ApplicationID' and elem.text:
#         app_ids.append(elem.text)
#         print(f"Found ApplicationID: {elem.text}")

# print(f"Total ApplicationID elements: {len(app_ids)}\n")

In [345]:
def run_pipeline(vin: str):
    """
    Complete pipeline: VIN -> All work-time applications and their parts.
    
    Returns a flat dict where keys are service names and values are lists of work items.
    Each work item contains all labor details plus a 'parts' dict (keyed by part_app_id).
    
    Output structure:
    {
        'Rack & Pinion Assembly R&R': [
            {
                'application_id': '244560030',
                'vehicle_id': '85215',
                'job_description': '...',
                'base_labor_time': '0.4',
                ... (all 20 labor fields),
                'parts': {
                    'part_app_id_1': {
                        'part_number': 'KG9Z 3504-H',
                        'price': '2718.18',
                        'manufacturer_name': 'Ford',
                        ...
                    },
                    'part_app_id_2': {...}
                }
            },
            { second_work_item_with_parts },
            ...
        ],
        'Steering Knuckle R&R': [ ... ],
        ...
    }
    """
    print(f"Starting pipeline for VIN: {vin}")
    
    # STEP 1: Vehicle lookup
    print("STEP 1: Vehicle lookup...")
    vehicle_data = step_1(vin)
    print(f"OK: Found: {vehicle_data['vehicles'][0]['make_name']} {vehicle_data['vehicles'][0]['model_name']} {vehicle_data['vehicles'][0]['year']}")
    
    # STEP 2: Get all work-time application IDs
    print("STEP 2: Get work-time summaries...")
    summaries = step2(vehicle_data)
    num_apps = len(summaries['application_ids'])
    print(f"OK: Found {num_apps} work-time applications")
    
    # STEP 3: Extract all work items from each application
    print("STEP 3: Extract work items...")
    labor_items = step3(summaries)
    print(f"OK: Processed {len(labor_items)} services")
    
    # STEP 4: Enrich each work item with its parts
    print("STEP 4: Enrich with parts data...")
    complete_data = step_4(labor_items)
    
    # Count total parts
    total_parts = 0
    for service_name, work_items in complete_data.items():
        for work_item in work_items:
            total_parts += len(work_item.get('parts', {}))
    
    print(f"OK: Found {total_parts} total parts")
    print(f"OK: Pipeline complete!")
    
    return complete_data

In [346]:

pipe_1 = run_pipeline(veh_vin["US"]["escape_2020"]) 
 

Starting pipeline for VIN: 3FA6P0D9XLR115438
STEP 1: Vehicle lookup...
Response Status: OK (200)

OK: Found: Ford Fusion 2020
STEP 2: Get work-time summaries...
OK: Found 5 work-time applications
STEP 3: Extract work items...
OK: Processed 5 services
STEP 4: Enrich with parts data...
OK: Found 92 total parts
OK: Pipeline complete!


In [354]:
pipe_1["Rack & Pinion Assembly R&R"][0]["parts"]

{'338614419': {'part_number': 'KG9Z 3504-H',
  'country_name': 'United States',
  'manufacturer_name': 'Ford',
  'effective_date': '2026-05-01T00:00:00',
  'is_current': 'true',
  'oepr_part_number': 'KG9Z 3504-H',
  'motor_part_number': 'KG9Z3504H',
  'net_core_price': '0.00',
  'category_name': 'Steering',
  'part_terminology_name': 'Rack and Pinion Assembly',
  'price': '2718.18',
  'return_old_part': 'T'},
 '338614420': {'part_number': 'KG9Z 3504-H',
  'country_name': 'United States',
  'manufacturer_name': 'Ford',
  'effective_date': '2026-05-01T00:00:00',
  'is_current': 'true',
  'oepr_part_number': 'KG9Z 3504-H',
  'motor_part_number': 'KG9Z3504H',
  'net_core_price': '0.00',
  'category_name': 'Steering',
  'part_terminology_name': 'Rack and Pinion Assembly',
  'price': '2718.18',
  'return_old_part': 'T'},
 '338614421': {'part_number': 'KG9Z 3504-H',
  'country_name': 'United States',
  'manufacturer_name': 'Ford',
  'effective_date': '2026-05-01T00:00:00',
  'is_current': 't

In [204]:

# # Test the complete pipeline
# if __name__ != "__main__":
#     print("
# " + "="*80)
#     print("TESTING COMPLETE PIPELINE")
#     print("="*80 + "
# ")
    
#     try:
#         # Run the pipeline with the test VIN
#         test_vin = veh_vin["US"]["escape_2020"]
#         print(f"Running pipeline for VIN: {test_vin}
# ")
        
#         results = run_pipeline(test_vin)
        
#         # Show summary
#         print("
# " + "="*80)
#         print("PIPELINE RESULTS SUMMARY")
#         print("="*80 + "
# ")
        
#         total_services = 0
#         total_work_items = 0
#         total_parts = 0
        
#         for service_dict in results:
#             for service_name, work_items in service_dict.items():
#                 total_services += 1
#                 total_work_items += len(work_items)
#                 for work_item in work_items:
#                     parts = work_item.get('parts', [])
#                     total_parts += len(parts)
#                     print(f"  Service: {service_name}")
#                     print(f"    Work Item ID: {work_item.get('application_id')}")
#                     print(f"    Description: {work_item.get('job_description', 'N/A')[:60]}...")
#                     print(f"    Parts: {len(parts)}")
#                     if parts:
#                         print(f"      First part: {parts[0].get('part_number')} ({parts[0].get('manufacturer_name')})")
#                     print()
        
#         print(f"
# FINAL SUMMARY:")
#         print(f"  Total services: {total_services}")
#         print(f"  Total work items: {total_work_items}")
#         print(f"  Total parts: {total_parts}")
        
#     except Exception as e:
#         import traceback
#         print(f"ERROR running pipeline: {e}")
#         traceback.print_exc()


In [ ]:
# Example: Convert pipeline results to Excel

# Run the complete pipeline
results = run_pipeline(veh_vin["US"]["escape_2020"])

# Convert to Excel with deduplication
file_path = pipeline_to_excel(results)

# Display summary of saved data
display_summary(file_path)


In [ ]:
# Individual helper functions (for advanced usage):

# 1. Setup output directory
output_dir = setup_output_directory()

# 2. Flatten results to rows
rows = flatten_pipeline_to_rows(results)
print(f"Generated {len(rows)} rows")

# 3. Convert to DataFrame
df = rows_to_dataframe(rows)
print(f"DataFrame shape: {df.shape}")

# 4. Load existing data
existing_df = load_existing_data(output_dir / "motor_data.xlsx")

# 5. Deduplicate
new_rows = deduplicate_rows(df, existing_df)

# 6. Append to Excel
append_to_excel(output_dir / "motor_data.xlsx", new_rows, existing_df)


In [ ]:
# Import required modules for Excel export
from pathlib import Path
import pandas as pd

In [ ]:
def setup_output_directory(base_dir: Path = None) -> Path:
    """
    Create output folder if it doesn't exist using pathlib.

    Returns:
        Path: Path to output directory
    """
    if base_dir is None:
        base_dir = Path.cwd()
    else:
        base_dir = Path(base_dir)

    output_dir = base_dir / "output"
    output_dir.mkdir(exist_ok=True, parents=True)

    print(f"Output directory ready: {output_dir}")
    return output_dir

In [ ]:
def flatten_pipeline_to_rows(pipeline_results: dict) -> list:
    """
    Convert nested pipeline structure to flat list of row dictionaries.
    One row per part with all work item context preserved.
    """
    rows = []

    # Iterate through each service
    for service_name, work_items in pipeline_results.items():
        # Iterate through each work item in this service
        for work_item in work_items:
            # Get the parts dict for this work item
            parts_dict = work_item.get('parts', {})

            # Iterate through each part in this work item
            for part_app_id, part_data in parts_dict.items():
                # Create a row combining work item + part data
                row = {
                    'service_name': service_name,
                    'part_app_id': part_app_id,
                    **work_item,  # Add all work item fields
                    **part_data   # Add all part fields
                }

                # Remove 'parts' dict from row (we extracted it already)
                row.pop('parts', None)

                rows.append(row)

    print(f"Flattened {len(rows)} rows from pipeline results")
    return rows

In [ ]:
def rows_to_dataframe(rows: list) -> pd.DataFrame:
    """
    Convert list of flat row dictionaries to DataFrame.
    Handles data type conversions for numeric and boolean fields.
    """
    if not rows:
        print("No rows to convert to DataFrame")
        return pd.DataFrame()

    # Create DataFrame from rows
    df = pd.DataFrame(rows)

    # Fields to convert to float
    float_fields = [
        'price', 'base_labor_time', 'all_labor_time',
        'base_labor_time_average', 'base_warranty_labor_time',
        'all_warranty_labor_time', 'additional_labor_time',
        'additional_warranty_labor_time'
    ]

    # Convert numeric fields
    for field in float_fields:
        if field in df.columns:
            df[field] = pd.to_numeric(df[field], errors='coerce')

    # Convert boolean fields
    if 'is_active' in df.columns:
        df['is_active'] = df['is_active'].map({'true': True, 'false': False})

    print(f"Created DataFrame with {len(df)} rows and {len(df.columns)} columns")
    return df

In [ ]:
def load_existing_data(file_path: Path) -> pd.DataFrame:
    """
    Load existing Excel file if it exists.
    Returns empty DataFrame if file doesn't exist.
    """
    # Check if file exists
    if file_path.exists():
        try:
            df = pd.read_excel(file_path)
            print(f"Loaded existing data: {len(df)} rows")
            return df
        except Exception as e:
            print(f"Error reading Excel file: {e}")
            return pd.DataFrame()
    else:
        print(f"No existing file found at {file_path}")
        return pd.DataFrame()

In [ ]:
def deduplicate_rows(new_df: pd.DataFrame, existing_df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove rows from new_df that already exist in existing_df.

    Deduplication Strategy:
    - Key columns: ['vehicle_id', 'application_id', 'part_app_id']
    - If all three columns match an existing row -> skip
    """
    # If no existing data, all new rows are unique
    if existing_df.empty:
        print(f"No existing data, all {len(new_df)} rows are new")
        return new_df

    # Create composite key from dedup columns
    dedup_cols = ['vehicle_id', 'application_id', 'part_app_id']

    # Check that all dedup columns exist
    missing_cols = [col for col in dedup_cols if col not in new_df.columns]
    if missing_cols:
        print(f"Warning: Missing dedup columns {missing_cols}, skipping deduplication")
        return new_df

    # Create composite keys
    new_keys = new_df[dedup_cols].astype(str).agg('_'.join, axis=1)
    existing_keys = existing_df[dedup_cols].astype(str).agg('_'.join, axis=1)

    # Find rows that don't exist in existing data
    new_rows_mask = ~new_keys.isin(existing_keys)
    new_rows = new_df[new_rows_mask]

    duplicates = len(new_df) - len(new_rows)
    print(f"Deduplication: {duplicates} duplicates removed, {len(new_rows)} new rows remain")

    return new_rows

In [ ]:
def append_to_excel(file_path: Path, new_df: pd.DataFrame, existing_df: pd.DataFrame) -> None:
    """
    Append new rows to Excel file.
    Combines existing + new data and writes to file.
    """
    # Combine existing + new data
    if existing_df.empty:
        combined_df = new_df.copy()
        print(f"Creating new file with {len(combined_df)} rows")
    else:
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        print(f"Combined data: {len(existing_df)} existing + {len(new_df)} new = {len(combined_df)} total rows")

    # Write to Excel
    try:
        combined_df.to_excel(file_path, index=False, sheet_name='Motor Data')
        print(f"Successfully saved to {file_path}")
    except Exception as e:
        print(f"Error writing to Excel: {e}")

In [ ]:
def pipeline_to_excel(
    pipeline_results: dict,
    file_name: str = "motor_data",
    output_dir: Path = None
) -> Path:
    """
    Main orchestration function.
    Converts pipeline results to DataFrame and saves to Excel with deduplication.

    Workflow:
    1. Setup output directory
    2. Flatten pipeline to rows
    3. Convert rows to DataFrame
    4. Load existing data (if file exists)
    5. Deduplicate new rows
    6. Append to Excel

    Args:
        pipeline_results: Output from run_pipeline()
        file_name: Name of Excel file without extension (default: "motor_data")
        output_dir: Output directory path (default: ./output)

    Returns:
        Path: Path to saved Excel file
    """
    print("\n" + "="*80)
    print("PIPELINE TO EXCEL CONVERSION")
    print("="*80 + "\n")

    # STEP 1: Setup output directory
    print("STEP 1: Setting up output directory...")
    output_directory = setup_output_directory(output_dir)
    file_path = output_directory / f"{file_name}.xlsx"
    print()

    # STEP 2: Flatten pipeline to rows
    print("STEP 2: Flattening pipeline results...")
    rows = flatten_pipeline_to_rows(pipeline_results)
    if not rows:
        print("ERROR: No rows generated from pipeline results")
        return file_path
    print()

    # STEP 3: Convert rows to DataFrame
    print("STEP 3: Converting to DataFrame...")
    new_df = rows_to_dataframe(rows)
    print()

    # STEP 4: Load existing data
    print("STEP 4: Loading existing data...")
    existing_df = load_existing_data(file_path)
    print()

    # STEP 5: Deduplicate rows
    print("STEP 5: Deduplicating rows...")
    unique_rows = deduplicate_rows(new_df, existing_df)
    print()

    # STEP 6: Append to Excel
    print("STEP 6: Saving to Excel...")
    if unique_rows.empty:
        print("No new rows to save")
    else:
        append_to_excel(file_path, unique_rows, existing_df)
    print()

    print("="*80)
    print(f"COMPLETE: Data saved to {file_path}")
    print("="*80 + "\n")

    return file_path

In [ ]:
def display_summary(file_path: Path) -> None:
    """
    Display summary of saved Excel file.
    """
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return

    df = pd.read_excel(file_path)

    print("\n" + "="*80)
    print("EXCEL FILE SUMMARY")
    print("="*80)
    print(f"File: {file_path}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nColumn Names:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")

    print(f"\nFirst 5 Rows:")
    print(df.head().to_string())
    print("="*80 + "\n")

In [ ]:
# USAGE EXAMPLE FOR EXCEL EXPORT
# ================================

# Step 1: Run the complete pipeline
# results = run_pipeline(veh_vin["US"]["escape_2020"])

# Step 2: Convert to Excel with one command
# file_path = pipeline_to_excel(results)

# Step 3: Display summary of saved data
# display_summary(file_path)

# Optional: Custom output directory
# file_path = pipeline_to_excel(results, output_dir=Path("custom_output"))

print("Excel export functions loaded successfully!")
print("Ready to use:")
print("  1. results = run_pipeline(vin)")
print("  2. file_path = pipeline_to_excel(results)")
print("  3. display_summary(file_path)")